# list_editions — detecção das edições a processar

Task `list_editions` do job `job_munic_full_refresh` (orquestrador): importa o `env_variables`, extrai as **chaves** de `munic_source_url_map` — a **fonte da verdade** dos anos disponíveis — e publica a lista como *task value* `years`, que o `for_each` do orquestrador lê (`{{tasks.list_editions.values.years}}`) para rodar o pipeline completo, uma edição por vez.

Não existe lista manual de anos em nenhum lugar: registrar uma nova edição = adicionar 1 linha no `munic_source_url_map`; o orquestrador a detecta sozinho na próxima execução.

In [0]:
%run ../environment/env_variables

In [0]:
# As edições a processar são as CHAVES de munic_source_url_map: o que está registrado
# no mapa é o que existe para baixar/processar. Ordem crescente (2019 antes de 2024,
# etc.) — o orquestrador executa uma por vez, em sequência (concurrency 1).
editions = sorted(str(year) for year in munic_source_url_map)
print(f"Edições registradas em munic_source_url_map: {editions}")

# Publica a lista como task value "years". Fora de um job (execução interativa),
# o set é inofensivo: o valor simplesmente não é consumido por ninguém.
dbutils.jobs.taskValues.set(key="years", value=editions)
print('Task value "years" publicado para o for_each do orquestrador.')